# REE and coal exploration

Load samples, prepare low/high labels, and evaluate a coal-chemistry classifier. Run cells from top to bottom.

## Setup and configuration

Dataset names, feature lists, reference values, label prices, and model settings live in `util/config.py`. Restart the kernel after editing utilities.

In [ ]:
import pandas as pd
from xgboost import XGBClassifier

from util import config as cfg
from util.data_preparation import (
    list_layers, load_layer, select_columns, fill_missing_ree,
    add_priority_labels, merge_samples,
)
from util.qualifiers import resolve_qualifiers
from util.modeling import prepare_train_test, evaluate_classifier
from util.plotting import plot_classifier_diagnostics

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
gdb_path = cfg.GDB_PATH

## Explore available layers

In [ ]:
layers = list_layers(gdb_path)
search_text = "data"
display(layers[layers["name"].str.contains(search_text, case=False, regex=False)])

## Load and select elemental data

Keep chemistry, qualifier flags, and sample context required downstream.

In [ ]:
element_data_raw = load_layer(gdb_path, cfg.ELEMENT_LAYER)
element_data = select_columns(element_data_raw, cfg.ELEMENT_COLS_NEEDED)
element_data.head()

## Prepare measurements and labels

Keep the existing half-limit and non-detection substitutions. Interpolate missing REEs only between observed values; Y is not interpolated. Split the price-weighted score into low/high quantiles.

In [ ]:
element_data = resolve_qualifiers(
    element_data, cfg.REY_ELEMENTS + cfg.FEATURE_ELEMENTS
)
element_data = fill_missing_ree(element_data, cfg.TARGETS)
element_data = add_priority_labels(
    element_data, cfg.LABEL_ELEMENTS, cfg.PRICE_PER_KG, cfg.CLASS_NAMES
)
element_data.head()

## Load coal quality and join samples

Join on `Sample_ID` and inspect match counts.

In [ ]:
coal_quality_raw = load_layer(gdb_path, cfg.QUALITY_LAYER)
coal_quality = select_columns(coal_quality_raw, cfg.COAL_QUALITY_COLS_NEEDED)
final_data = merge_samples(element_data, coal_quality)
print(f"Final merged shape: {final_data.shape}")
final_data.head()

## Train and evaluate

Preserve the stratified random split and model settings. Encode categorical inputs after splitting. This score measures the existing random holdout, not performance in unseen regions.

In [ ]:
X_train, X_test, y_train, y_test = prepare_train_test(
    final_data, cfg.FEATURE_COLS, cfg.CATEGORICAL_COLS,
    cfg.TEST_SIZE, cfg.RANDOM_STATE,
)
model = XGBClassifier(**cfg.MODEL_PARAMS)
model.fit(X_train, y_train)
preds = evaluate_classifier(model, X_test, y_test, cfg.CLASS_NAMES)

## Inspect predictions

Review class metrics, ROC and precision-recall curves, probabilities, and feature importance.

In [ ]:
figures = plot_classifier_diagnostics(
    model, X_test, y_test, X_train.columns, cfg.CLASS_NAMES
)